# Day 22 — Frozen Final Model
This notebook reproduces the candidate comparison and held-out evaluation. Timestamps are synthetic.

In [ ]:
import pandas as pd, numpy as np, joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df=pd.read_csv('water_potability_temporal_synthetic.csv')
df['timestamp']=pd.to_datetime(df['timestamp'],errors='coerce')
sensor_cols=[c for c in df.columns if c not in ['timestamp','Potability']]
data=df.dropna(subset=['timestamp']).sort_values('timestamp',kind='stable').reset_index(drop=True)
data['target_next_ph']=data['ph'].shift(-1)
data=data.dropna(subset=['target_next_ph']).reset_index(drop=True)
n=len(data); a=int(n*.70); b=int(n*.85)
train,val,test=data.iloc[:a],data.iloc[a:b],data.iloc[b:]
print('Rows:',len(train),len(val),len(test))

In [ ]:
def linear(): return Pipeline([('imputer',SimpleImputer(strategy='median')),('model',LinearRegression())])
def forest(): return Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestRegressor(n_estimators=150,max_depth=12,min_samples_leaf=4,max_features=.8,random_state=42,n_jobs=-1))])
candidates=[('Linear Regression pH-only',['ph'],linear()),('Random Forest pH-only',['ph'],forest()),('Random Forest full features',sensor_cols,forest())]
rows=[]
for name,cols,m in candidates:
 m.fit(train[cols],train.target_next_ph); p=m.predict(val[cols])
 rows.append({'candidate':name,'MAE':mean_absolute_error(val.target_next_ph,p),'RMSE':np.sqrt(mean_squared_error(val.target_next_ph,p))})
p=val.ph.fillna(train.ph.median()).to_numpy()
rows.append({'candidate':'Persistence current pH','MAE':mean_absolute_error(val.target_next_ph,p),'RMSE':np.sqrt(mean_squared_error(val.target_next_ph,p))})
pd.DataFrame(rows).sort_values('RMSE')

In [ ]:
final=linear(); development=pd.concat([train,val],ignore_index=True)
final.fit(development[['ph']],development.target_next_ph)
test_pred=final.predict(test[['ph']])
print('Test MAE:',mean_absolute_error(test.target_next_ph,test_pred))
print('Test RMSE:',np.sqrt(mean_squared_error(test.target_next_ph,test_pred)))
joblib.dump(final,'Day22_Final_Linear_Regression_Pipeline_reproduced.joblib')

The latest 15% test partition is evaluated once after selection. Synthetic timestamps mean these are exploratory pseudo-temporal results, not real deployment evidence.